# FoodScholar — Quickstart

Three cells from zero to a populated graph using your local corpus + pre-computed NER/NEL.

```
pip install -e '.[elastic,neo4j,ontology]'
# local services (matching the inline config below):
docker run -d -p 9200:9200 -e discovery.type=single-node \
    -e xpack.security.enabled=false elasticsearch:8.13.0
docker run -d -p 7687:7687 -p 7474:7474 \
    -e NEO4J_AUTH=neo4j/change-me neo4j:5
```

This notebook does **not** run GLiNER. Pre-computed annotations from
`data/foodscholar/ner/*.csv` are loaded and attached to the chunks — fast,
deterministic, no model downloads. The "Under the hood" appendix at the
bottom shows how to do the same end-to-end with GLiNER + HNSW when no
pre-computed output is on disk.

## 1. Configure

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from foodscholar import FoodScholar
from foodscholar.logging import configure_logging
configure_logging(level="INFO")

CORPUS_DIR = REPO_ROOT / "data" / "foodscholar" / "corpus"   # chunks_*.csv
NEL_DIR    = REPO_ROOT / "data" / "foodscholar" / "ner"      # nel_chunks_*.csv
FOODON_OWL = REPO_ROOT / "data" / "foodon.owl"

fs = FoodScholar.from_config({
    "corpus": {
        "chunks_path": str(CORPUS_DIR),
        "annotated_snapshot_path": str(REPO_ROOT / "data" / "annotated.parquet"),
    },
    "ontology": {
        "foodon_path": str(FOODON_OWL),
        "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
        "prefix_filter": ["FOODON:"],
    },
    "storage": {
        "chunk_store": {
            "backend": "elastic",
            "url": "http://localhost:9200",
            "index": "foodscholar_chunks",
        },
        "graph_store": {
            "backend": "neo4j",
            "url": "bolt://localhost:7687",
            "user": "neo4j",
            "password": "change-me",
        },
    },
})
fs.info()

## 2. Init stores & ingest

`fs.init()` creates the Elastic index and the Neo4j unique constraints — idempotent.

`fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR)` reads every `chunks_*.csv`, attaches the matching annotations from `nel_*.csv`, and upserts to Elastic. **No GLiNER, no HNSW, no embedding** — this is fast and works without the `[annotate]` extra installed.

Chunk-text embeddings are populated in a separate step (§3) so you can iterate on annotations without re-encoding.

In [ ]:
fs.init()
# Abstracts are excluded by design — the prototype's `nel_*.csv` set has no
# nel_*abstracts*.csv file, so a chunk loaded from chunks_abstracts.csv would
# arrive with zero NEL annotations. Adding 600k+ such chunks would inflate the
# corpus pool without contributing any support to Layer A. Restore them only
# after the abstract NEL pass is re-run (or after switching to GLiNER+HNSW
# via `fs.load_and_annotate(CORPUS_DIR)`).
meta = fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR, ignore_source_types=['abstract'])
print(meta)

## 3. Embed (optional, for vector search)

Run this once chunks are in the store and you want kNN search to work. It builds the
production embedder — `SourceTypeRouter(SPECTER2 for abstracts, BGE-large for guides /
textbooks)` — lazily on the first call (~1.7 GB model load) and patches each chunk's
`embedding` + `embedding_model` fields in Elastic via a scoped `_update` (annotations untouched).

Default `only_missing=True` skips chunks that already carry a real vector, so re-runs after
adding new chunks only encode what's new.

Requires the `[annotate]` extra (`pip install -e '.[annotate]'`).

In [ ]:
meta = fs.embed()
print(meta)

## 4. Build & explore entities

Linked entities are first-class. `fs.build_entities()` walks the chunk store, dedupes every `EntityLink` by `ontology_id` (FOODON, CHEBI, GAZ, ...), aggregates `mention_count` / `chunk_count` / sample `chunk_ids`, enriches FoodOn ids with label + synonyms + ancestors from the loaded ontology, and writes everything to:

- **Elastic** — a dedicated `foodscholar_chunks_entities` index for fast lookup + search.
- **Neo4j** — `(:Entity)` nodes plus `(:Chunk)-[:MENTIONS {confidence, method}]->(:Entity)` edges.

Then `fs.entities` is the user-facing handle: `.list(prefix="FOODON")`, `.get(id)`, `.search("olive")`, `.chunks_for(id)`.

In [ ]:
meta = fs.build_entities()
print(meta)


In [ ]:
print(f"\ntotal entities: {len(fs.entities)}")
# Top FoodOn entities by chunk_count.
for ent in fs.entities.list(k=8):
    print(f"  {ent.ontology_id:24s}  {ent.label!r:40s}  chunks={ent.chunk_count}")

# Lexical search over labels + synonyms.
print("\nsearch 'olive':")
for ent in fs.entities.search("olive", k=3):
    print(f"  {ent.ontology_id}  {ent.label}")

# Reverse lookup: chunks that mention a specific entity.
top = fs.entities.list(prefix="FOODON", k=1)
if top:
    print(f"\nchunks mentioning {top[0].ontology_id} ({top[0].label}):")
    for c in fs.entities.chunks_for(top[0].ontology_id, k=3):
        print(f"  {c.chunk_id}  {c.text[:80]}")

## 5. Build Layer A (backbone)

Layer A is FoodOn (and the other OBO ontologies we link to) projected onto **your** corpus — not a copy of the raw ontology. For every FoodOn class, count how many chunks have at least one link to that class or any of its descendants, then prune ruthlessly: blacklist organizational classes, drop everything below `min_support`, lift shelves above `max_depth` to a closer ancestor, and collapse single-child chains into their leaves (recording the dropped ids in `see_also`).

Non-foods facets (`health`, `nutrients`, `dietary_patterns`, `allergies`) derive from `Mention.entity_type` on EntityLinks. The prototype's pre-computed NEL data sets every `entity_type` to `"other"`, so on this corpus those facets land as **stub roots only** — they'll fill in once chunks are re-annotated with GLiNER. Sustainability is always a stub root (no entity_type maps to it).

Layer A here is **projection only**. Chunk attachment (`fs.attach()`) is the next phase.

Re-running `fs.build_layer_a()` is idempotent: every call clears the prior `:Shelf` projection from the graph store before writing the new one, so you can re-tune `min_support` / `blacklist_terms` / `facet_overrides` freely without stale ghost shelves piling up across iterations.

In [ ]:
# Tune for the current corpus — defaults are conservative.
fs.config.layer_a.min_support = 25
fs.config.layer_a.max_depth = 6
fs.config.layer_a.collapse_single_child_chains = True

# Echo the resolved foods-facet config so the prune cascade's parameters
# are visible at a glance. Override anything per-facet via
# `fs.config.layer_a.facet_overrides["foods"] = FacetConfig(...)`.
_foods = fs.config.layer_a.resolve_facet('foods')
print(f'foods config: min_support={_foods.min_support}, max_depth={_foods.max_depth}, '
      f'collapse={_foods.collapse_single_child_chains}, '
      f'umbrella=({_foods.umbrella_direct_share_max}, {_foods.umbrella_lifted_share_min}, '
      f'min_count={_foods.umbrella_min_count}), '
      f'blacklist={_foods.blacklist_terms}')

meta = fs.build_layer_a()
print(meta)

In [ ]:
from collections import Counter

shelves = fs.graph_store.list_shelves()
by_facet = Counter(s.facet for s in shelves)
print('shelves per facet:')
for facet, n in sorted(by_facet.items(), key=lambda kv: -kv[1]):
    print(f'  {facet:18s} {n}')

foods = [s for s in shelves if s.facet == 'foods']
print(f'\ntop 10 foods shelves by chunk_count:')
for s in sorted(foods, key=lambda s: -s.chunk_count)[:10]:
    direct_share = s.support_direct / s.chunk_count if s.chunk_count else 0
    print(f'  {s.label[:40]:40s}  {s.chunk_count:>5d}  (direct={s.support_direct} lifted={s.support_lifted} direct_share={direct_share:.3f})')

print(f'\nbottom 5 surviving foods shelves:')
for s in sorted(foods, key=lambda s: s.chunk_count)[:5]:
    print(f'  {s.label[:40]:40s}  {s.chunk_count:>5d}')

depths = Counter(s.depth for s in foods)
print(f'\nfoods depth distribution:')
for d in sorted(depths):
    print(f'  depth {d}: {depths[d]}')

# Inflation diagnostic — shelves whose support is mostly lifted (per the spec,
# >0.9 means a deep dense subtree funneled into one ancestor). After the
# umbrella rule lands these should be rare — surviving inflated shelves are
# real navigation roots (e.g. `food product` itself) rather than scaffolding.
inflated = [s for s in foods if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9]
if inflated:
    print(f'\n{len(inflated)} potentially-inflated shelves (support_lifted/chunk_count > 0.9):')
    for s in sorted(inflated, key=lambda s: -s.chunk_count)[:5]:
        ratio = s.support_lifted / s.chunk_count
        direct_share = s.support_direct / s.chunk_count
        print(f'  {s.label[:40]:40s}  lifted_ratio={ratio:.2f}  direct_share={direct_share:.3f}  count={s.chunk_count}')
else:
    print('\nno inflation flags.')

# Orphan diagnostic — shelves at depth 0 that aren't the stub roots. After
# the umbrella rule lands these are the navigation roots that no surviving
# parent could be found for. Whitelisting the foods root (FOODON:00001002)
# typically reduces this to 1-2.
depth0_foods = [s for s in foods if s.depth == 0]
print(f'\n{len(depth0_foods)} foods shelves at depth 0 (roots):')
for s in sorted(depth0_foods, key=lambda s: -s.chunk_count)[:10]:
    print(f'  {s.label[:40]:40s}  count={s.chunk_count}  foodon_id={s.foodon_id}')

### 5b. Why these thresholds? — parameter sweep

The three knobs the prune cascade exposes (`min_support`, `umbrella_direct_share_max`, `umbrella_min_count`) are empirical, not derived. This cell sweeps each one independently against the live corpus and plots the trade-off curves, then a 2D heatmap of the two umbrella knobs together. The chosen values are marked.

Cheap: support is collected once over the chunk store, then `prune(...)` re-runs per config combo without touching Neo4j. Runs in seconds even on a multi-thousand-chunk corpus.

In [ ]:
import matplotlib.pyplot as plt
from foodscholar.layer_a.propagate import collect_support
from foodscholar.layer_a.prune import prune
from foodscholar.config import FacetConfig, LayerAConfig

# Collect support once — every sweep point re-prunes against this in-memory
# table. ~seconds for ~13k chunks; the bottleneck would be Neo4j writes,
# which we skip.
def _chunk_iter():
    for batch in fs.chunk_store.iter_chunks():
        yield from batch

base = fs.config.layer_a.resolve_facet('foods')
support = collect_support(_chunk_iter(), fs.ontology,
                          min_link_confidence=base.min_link_confidence,
                          facet='foods')
print(f'support table: {len(support.with_descendants)} terms, '
      f'{sum(support.direct.values())} direct mentions')

def _resolve_with(**overrides):
    """Build a fully-resolved foods FacetConfig by overlaying overrides on the
    live `fs.config.layer_a`. Goes through `resolve_facet` so we touch only the
    public config surface."""
    base_cfg = fs.config.layer_a
    layer_a = LayerAConfig.model_validate(base_cfg.model_dump())
    layer_a.facet_overrides = dict(layer_a.facet_overrides)
    layer_a.facet_overrides['foods'] = FacetConfig(**overrides)
    return layer_a.resolve_facet('foods')

def _summarize(shelves: list) -> dict:
    inflated = sum(1 for s in shelves
                   if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9)
    roots = sum(1 for s in shelves if s.parent_shelf_id is None)
    return {'n_shelves': len(shelves), 'n_inflated': inflated, 'n_roots': roots}

# Run the three single-knob sweeps.
ms_grid = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200]
ds_grid = [0.0, 0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]
mc_grid = [10, 25, 50, 75, 100, 150, 200, 300, 500, 1000]

ms_data = [(ms, _summarize(prune(support, fs.ontology, _resolve_with(min_support=ms), 'foods'))) for ms in ms_grid]
ds_data = [(ds, _summarize(prune(support, fs.ontology, _resolve_with(umbrella_direct_share_max=ds), 'foods'))) for ds in ds_grid]
mc_data = [(mc, _summarize(prune(support, fs.ontology, _resolve_with(umbrella_min_count=mc), 'foods'))) for mc in mc_grid]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def _plot(ax, data, knob_name, chosen, log_x=False):
    xs = [d[0] for d in data]
    n_shelves = [d[1]['n_shelves'] for d in data]
    n_inflated = [d[1]['n_inflated'] for d in data]
    ax.plot(xs, n_shelves, 'o-', label='surviving shelves')
    ax.plot(xs, n_inflated, 's--', label='inflated (>0.9 lifted)')
    ax.axvline(chosen, linestyle=':', label=f'chosen = {chosen}')
    ax.set_title(knob_name)
    ax.set_xlabel(knob_name)
    ax.set_ylabel('shelves')
    ax.grid(True)
    if log_x:
        ax.set_xscale('log')
    ax.legend()

_plot(axes[0], ms_data, 'min_support', base.min_support, log_x=True)
_plot(axes[1], ds_data, 'umbrella_direct_share_max', base.umbrella_direct_share_max)
_plot(axes[2], mc_data, 'umbrella_min_count', base.umbrella_min_count, log_x=True)

fig.suptitle('Layer A — single-knob parameter sweep')
plt.tight_layout()
plt.show()

# Print the chosen-point summary as numeric grounding for the chart.
chosen_cfg = _resolve_with()  # all defaults
chosen_shelves = prune(support, fs.ontology, chosen_cfg, 'foods')
chosen = _summarize(chosen_shelves)
print(f"\nat chosen config (min_support={base.min_support}, "
      f"direct_share_max={base.umbrella_direct_share_max}, "
      f"min_count={base.umbrella_min_count}):")
print(f"  shelves:               {chosen['n_shelves']}")
print(f"  inflated (>0.9 lifted): {chosen['n_inflated']}")
print(f"  orphan roots:           {chosen['n_roots']}")

In [ ]:
import numpy as np

# Joint sweep: direct_share_max × min_count. Heatmap colors the joint effect.
ds_axis = [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30]
mc_axis = [25, 50, 75, 100, 150, 200, 300, 500]

grid_shelves = np.zeros((len(mc_axis), len(ds_axis)))
grid_inflated = np.zeros((len(mc_axis), len(ds_axis)))
for i, mc in enumerate(mc_axis):
    for j, ds in enumerate(ds_axis):
        cfg = _resolve_with(umbrella_direct_share_max=ds, umbrella_min_count=mc)
        shelves = prune(support, fs.ontology, cfg, 'foods')
        s = _summarize(shelves)
        grid_shelves[i, j] = s['n_shelves']
        grid_inflated[i, j] = s['n_inflated']
clean = grid_shelves - grid_inflated

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def _heatmap(ax, grid, cmap, title):
    im = ax.imshow(grid, aspect='auto', origin='lower', cmap=cmap)
    ax.set_xticks(range(len(ds_axis)))
    ax.set_xticklabels([f'{d:g}' for d in ds_axis])
    ax.set_yticks(range(len(mc_axis)))
    ax.set_yticklabels(mc_axis)
    ax.set_xlabel('umbrella_direct_share_max')
    ax.set_ylabel('umbrella_min_count')
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    for i in range(len(mc_axis)):
        for j in range(len(ds_axis)):
            ax.text(j, i, f'{grid[i, j]:.0f}', ha='center', va='center')

_heatmap(axes[0], clean, 'viridis', 'Clean shelves (surviving − inflated)')
_heatmap(axes[1], grid_inflated, 'Reds', 'Inflated shelves')

# Mark the chosen config in both heatmaps.
if base.umbrella_direct_share_max in ds_axis and base.umbrella_min_count in mc_axis:
    j_chosen = ds_axis.index(base.umbrella_direct_share_max)
    i_chosen = mc_axis.index(base.umbrella_min_count)
    for ax in axes:
        ax.add_patch(plt.Rectangle((j_chosen - 0.5, i_chosen - 0.5), 1, 1,
                                   fill=False, edgecolor='black', linewidth=2))

fig.suptitle('Layer A — joint sweep of the two umbrella knobs')
plt.tight_layout()
plt.show()

### 5c. What happens when you raise `max_depth`?

The depth cap doesn't drop deep terms — it **lifts** them to the nearest surviving ancestor at depth ≤ cap (see [prune.py](../src/foodscholar/layer_a/prune.py) and the §7.0 PROGRESS entry on op order). So raising the cap doesn't *add* terms; it lets terms that were already there land deeper instead of bunching up at the cap.

Three things should shift as the cap rises:

1. **Surviving shelves go up** — more terms keep their natural ontology depth instead of getting clipped + collapsed away.
2. **`support_lifted` redistributes downward** — chunks that were lifting onto a shallow ancestor now find a more specific match closer to the leaf. Inflation drops on shelves that were absorbing everything from below.
3. **Median direct-share rises** — deep shelves get their own chunks instead of letting them lift up.

This sweep is **projection-only** (same recipe as §5b): support is collected once, `prune()` re-runs per cap value, no Neo4j writes. Re-running `fs.attach()` per cap would also redistribute *edges*, but the projection-side picture is cheap and tells you whether raising the cap is even worth doing.

In [ ]:
from statistics import median

# Depth sweep — `support` and `_resolve_with` are still in scope from §5b.
depth_grid = [3, 4, 5, 6, 7, 8, 9]

def _depth_summary(shelves):
    if not shelves:
        return {'n_shelves': 0, 'n_inflated': 0, 'median_direct_share': 0.0,
                'depth_dist': {}, 'shelves_at_or_above_cap': 0}
    inflated = sum(1 for s in shelves
                   if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9)
    direct_shares = [s.support_direct / s.chunk_count
                     for s in shelves if s.chunk_count > 0]
    depth_dist = Counter(s.depth for s in shelves)
    cap = max(depth_dist) if depth_dist else 0
    return {
        'n_shelves': len(shelves),
        'n_inflated': inflated,
        'median_direct_share': median(direct_shares) if direct_shares else 0.0,
        'depth_dist': dict(depth_dist),
        'shelves_at_or_above_cap': depth_dist.get(cap, 0),
    }

depth_data = []
for d in depth_grid:
    cfg = _resolve_with(max_depth=d)
    shelves = prune(support, fs.ontology, cfg, 'foods')
    depth_data.append((d, _depth_summary(shelves)))

xs = [d[0] for d in depth_data]
n_shelves = [d[1]['n_shelves'] for d in depth_data]
n_inflated = [d[1]['n_inflated'] for d in depth_data]
median_share = [d[1]['median_direct_share'] for d in depth_data]

CHOSEN_DEPTH = fs.config.layer_a.max_depth

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: shelves + inflation vs depth cap.
ax = axes[0]
ax.plot(xs, n_shelves, 'o-', label='surviving shelves')
ax.plot(xs, n_inflated, 's--', label='inflated (>0.9 lifted)')
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('max_depth: surviving + inflated shelves')
ax.set_xlabel('max_depth')
ax.set_ylabel('shelves')
ax.grid(True)
ax.legend()

# Panel 2: median direct-share vs depth cap.
ax = axes[1]
ax.plot(xs, median_share, 'o-', label='median direct-share')
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('median direct-share')
ax.set_xlabel('max_depth')
ax.set_ylabel('median support_direct / chunk_count')
ax.set_ylim(0, 1.05)
ax.grid(True)
ax.legend()

# Panel 3: depth distribution as stacked bars.
ax = axes[2]
all_depths = sorted({d for _, s in depth_data for d in s['depth_dist']})
bottoms = [0] * len(xs)
for depth in all_depths:
    heights = [s[1]['depth_dist'].get(depth, 0) for s in depth_data]
    ax.bar(xs, heights, bottom=bottoms, label=f'depth {depth}')
    bottoms = [b + h for b, h in zip(bottoms, heights, strict=True)]
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('shelf count by projection depth')
ax.set_xlabel('max_depth')
ax.set_ylabel('shelves')
ax.grid(True)
ax.legend(title='projection depth', ncol=2)

fig.suptitle('Layer A — depth-cap sweep')
plt.tight_layout()
plt.show()

# Numeric grounding — print the table so the chart values are auditable.
print(f'\n{"depth":>5s}  {"shelves":>7s}  {"inflated":>8s}  {"med_direct":>10s}  '
      f'{"shelves_at_cap":>14s}')
for d, s in depth_data:
    mark = '  *' if d == CHOSEN_DEPTH else ''
    print(f'{d:>5d}  {s["n_shelves"]:>7d}  {s["n_inflated"]:>8d}  '
          f'{s["median_direct_share"]:>10.3f}  {s["shelves_at_or_above_cap"]:>14d}{mark}')

### 5d. Why does so much support land at the synthetic root? — NEL coverage audit

The synthetic `Foods` root carries `support_lifted = chunk_count` by construction — every chunk reaching the facet contributes through lifting since nothing in any ontology resolves to the root itself. But the *size* of that number relative to the total corpus is informative: a high root count means many chunks failed to find a specific shelf, either because they have no FOODON links at all, or because all their links got umbrella-pruned.

This cell walks the chunk store once and reports:

- **NEL coverage**: how many chunks have ≥1 `foodon_id`, how many have only non-FOODON `entity_links`, how many have no links at all.
- **Sample of zero-foodon chunks**: 5 random texts so you can eyeball whether they're nutrition-y-but-not-food (expected) or food-content-the-NEL-missed (a real coverage gap).

In [ ]:
import random
from collections import Counter

n_total = 0
n_with_foodon = 0
n_only_other_obo = 0  # has entity_links but zero FOODON ids
n_no_links = 0
prefix_counter: Counter = Counter()
zero_foodon_samples: list = []

for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        n_total += 1
        has_foodon = bool(chunk.foodon_ids)
        has_any_link = bool(chunk.entity_links)
        if has_foodon:
            n_with_foodon += 1
        elif has_any_link:
            n_only_other_obo += 1
            for link in chunk.entity_links:
                oid = link.ontology_id
                prefix = oid.split(':', 1)[0] if ':' in oid else '(none)'
                prefix_counter[prefix] += 1
        else:
            n_no_links += 1

        # Keep a reservoir sample of zero-foodon chunks for hand inspection.
        if not has_foodon and len(zero_foodon_samples) < 200:
            zero_foodon_samples.append(chunk)

print(f'chunks total:                  {n_total:>6d}  (100.0%)')
print(f'  ≥1 FOODON id (well-placed):  {n_with_foodon:>6d}  ({100*n_with_foodon/n_total:>5.1f}%)')
print(f'  only non-FOODON entity_links: {n_only_other_obo:>6d}  ({100*n_only_other_obo/n_total:>5.1f}%)')
print(f'  zero entity links at all:     {n_no_links:>6d}  ({100*n_no_links/n_total:>5.1f}%)')

print(f'\nontology prefixes seen on chunks WITHOUT a FOODON id ({len(prefix_counter)} distinct):')
for prefix, n in prefix_counter.most_common(10):
    print(f'  {prefix:15s}  {n:>6d}')

print(f'\n5 random zero-foodon chunk texts (hand-check whether they should have had a FOODON link):')
random.seed(0)
for chunk in random.sample(zero_foodon_samples, k=min(5, len(zero_foodon_samples))):
    has_other = bool(chunk.entity_links)
    marker = '  [has non-FOODON links]' if has_other else '  [no links at all]'
    print(f'\n  {chunk.chunk_id}{marker}')
    print(f'    {chunk.text[:240].replace(chr(10), " ")}{"..." if len(chunk.text) > 240 else ""}')

### 5e. Sanity gate — 50 random FOODON-linked chunks

Pick 50 random chunks that have ≥1 FOODON link and print the text + the projected shelf labels each one rolls up to. The shelves shown mirror what `fs.attach()` writes in §6 — for every `foodon_id` on the chunk, this finds the deepest surviving shelf whose `foodon_id` is that term OR a surviving ancestor of it. Run this audit *before* `fs.attach()` to catch projection bugs early; run it again after to spot any drift between the resolver here and the real one.

Read 5–10 of these by hand. The question to answer: **is each chunk genuinely about the shelf it lands on?** Common failure modes to look for:
- NEL drift: a chunk mentioning "milky way" got linked to `cow milk`.
- Over-broad attachment: a chunk explicitly about kale ends up on `vegetable` instead of `kale` because the more specific shelf got pruned.
- Off-topic match: text is about cardiology but linked to `fat` via "fat embolism".

In [ ]:
import random

# Index surviving shelves three ways:
#   shelf_by_foodon  : direct foodon_id → shelf  (chunk linked to this term)
#   shelf_by_seealso : collapsed foodon_id → shelf  (term collapsed into
#                                                    survivor; see_also)
# A chunk's mention of a collapsed term legitimately attaches to the
# survivor — same as a lifted ancestor walk, just recorded differently.
shelves = fs.graph_store.list_shelves()
shelf_by_foodon = {s.foodon_id: s for s in shelves if s.foodon_id is not None}
shelf_by_seealso = {fid: s for s in shelves for fid in s.see_also}

def project_chunk(foodon_ids):
    """Returns (attached_shelves, attachment_details) where details says how
    each foodon_id reached its shelf: 'direct' / 'collapsed' / 'lifted' / None.
    Mirrors what fs.attach() will compute once it lands."""
    attached = set()
    details = []  # list of (foodon_id, label, mode, shelf_label_or_none)
    for fid in foodon_ids:
        label = fs.ontology.id_to_label(fid) or fid
        # 1. direct: shelf with this foodon_id
        if fid in shelf_by_foodon:
            sh = shelf_by_foodon[fid]
            attached.add(sh.label)
            details.append((fid, label, 'direct', sh.label))
            continue
        # 2. collapsed: term was absorbed into a surviving descendant via see_also
        if fid in shelf_by_seealso:
            sh = shelf_by_seealso[fid]
            attached.add(sh.label)
            details.append((fid, label, 'collapsed', sh.label))
            continue
        # 3. lifted: deepest surviving ancestor (umbrella/blacklist drops left a gap)
        ancestors = fs.ontology.id_to_ancestors(fid) if fs.ontology else []
        best = None
        best_depth = -1
        for anc in ancestors:
            sh = shelf_by_foodon.get(anc) or shelf_by_seealso.get(anc)
            if sh is not None and sh.depth > best_depth:
                best = sh
                best_depth = sh.depth
        if best is not None:
            attached.add(best.label)
            details.append((fid, label, 'lifted', best.label))
        else:
            details.append((fid, label, 'orphan', None))
    return sorted(attached), details

# Collect chunks with ≥1 FOODON id.
candidates = []
for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        if chunk.foodon_ids:
            candidates.append(chunk)
        if len(candidates) >= 5000:  # cap upstream sampling cost
            break
    if len(candidates) >= 5000:
        break

# Corpus-wide counts: how many chunks with foodon_ids end up orphaned?
# (This is the systemic-bug check the audit asked for.)
n_orphaned = 0
n_collapsed_only = 0
n_lifted_only = 0
for chunk in candidates:
    _, details = project_chunk(chunk.foodon_ids)
    modes = {d[2] for d in details}
    if modes == {'orphan'}:
        n_orphaned += 1
    elif 'direct' not in modes and 'collapsed' in modes and 'lifted' not in modes:
        n_collapsed_only += 1
    elif 'direct' not in modes and 'lifted' in modes and 'collapsed' not in modes:
        n_lifted_only += 1

print(f'corpus-wide attachment audit on {len(candidates)} FOODON-linked chunks:')
print(f'  fully orphaned (no shelf):           {n_orphaned}  ← real bugs')
print(f'  reached shelf only via collapse:     {n_collapsed_only}')
print(f'  reached shelf only via lifting:      {n_lifted_only}')
if n_orphaned > 50:
    print(f'  ⚠ {n_orphaned} orphaned chunks is high; investigate systematic gap')
elif n_orphaned > 0:
    print(f'  ({n_orphaned} orphans likely from terms whose ancestry exits FOODON via non-food OBOs)')

# Sample 50 + show 10 with attachment annotations.
random.seed(0)
sample = random.sample(candidates, k=min(50, len(candidates)))
print(f'\n--- sample of {min(10, len(sample))} chunks (raise [:10] to [:50] for full audit) ---\n')
for i, chunk in enumerate(sample[:10]):
    attached, details = project_chunk(chunk.foodon_ids)
    # Group details by mode for compact display.
    direct_lines  = [f'{lbl!r}' for _, lbl, mode, _ in details if mode == 'direct']
    collapsed_lines = [f'{lbl!r} → {target}' for _, lbl, mode, target in details if mode == 'collapsed']
    lifted_lines = [f'{lbl!r} → {target}' for _, lbl, mode, target in details if mode == 'lifted']
    orphan_lines = [f'{lbl!r}' for _, lbl, mode, _ in details if mode == 'orphan']

    print(f'[{i+1}] {chunk.chunk_id}')
    print(f'    attached shelves:      {attached}')
    if direct_lines:
        print(f'    foodon → direct:       {direct_lines}')
    if collapsed_lines:
        print(f'    foodon → collapsed:    {collapsed_lines}')
    if lifted_lines:
        print(f'    foodon → lifted:       {lifted_lines}')
    if orphan_lines:
        print(f'    foodon → orphan:       {orphan_lines}  ← cannot reach any shelf')
    text = chunk.text.replace('\n', ' ')
    print(f'    text: {text[:220]}{"..." if len(text) > 220 else ""}')
    print()

# Drift candidates: (surface, ontology_id) pairs across the sample that
# repeat ≥ 3 times. These are candidates to add to
# `fs.config.layer_a.link_blocklist` if the audit shows they're
# semantically wrong. Surface forms are case-insensitive in the blocklist.
from collections import Counter
drift_counter: Counter = Counter()
for chunk in sample:
    for link in chunk.entity_links:
        if link.ontology_id.startswith('FOODON:'):
            drift_counter[(link.mention.text.lower(), link.ontology_id)] += 1

frequent = [(pair, n) for pair, n in drift_counter.most_common() if n >= 3]
if frequent:
    print(f'\n--- (surface, ontology_id) pairs appearing ≥3 times across {len(sample)} sampled chunks ---')
    print('add to link_blocklist if semantically wrong:')
    for (surface, oid), n in frequent[:15]:
        label = fs.ontology.id_to_label(oid) or oid
        print(f'  {n:>3d}× ({surface!r:>30}, {oid:25s}) → label={label!r}')
else:
    print('\nno (surface, ontology_id) pair repeats ≥3 times — no obvious drift in this sample.')


### 5f. Sanity gate — 20 chunks with non-FOODON links only

These are the chunks that reached the foods facet only via the synthetic root (no FOODON id, but ≥1 entity_link of some other prefix). The question: **does the text contain a food term that should have linked but didn't?** That's a NEL coverage gap — the upstream linker missed a food mention.

If 5+ of 20 sampled chunks have an obvious missed food term, re-annotation with GLiNER is the right next step. If most are nutrition-y-but-not-food (papers about "macronutrient distribution", clinical population studies, etc.), the synthetic-root lift is expected and harmless.

In [ ]:
candidates_nonfoodon = []
for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        if not chunk.foodon_ids and chunk.entity_links:
            candidates_nonfoodon.append(chunk)
        if len(candidates_nonfoodon) >= 5000:
            break
    if len(candidates_nonfoodon) >= 5000:
        break

random.seed(1)
sample_nf = random.sample(candidates_nonfoodon, k=min(20, len(candidates_nonfoodon)))
print(f'sampled {len(sample_nf)} of {len(candidates_nonfoodon)} non-FOODON-only chunks scanned\n')

for i, chunk in enumerate(sample_nf):
    surface_forms = [link.mention.text for link in chunk.entity_links[:5]]
    prefixes = sorted({link.ontology_id.split(':', 1)[0] for link in chunk.entity_links})
    print(f'[{i+1}] {chunk.chunk_id}')
    print(f'    linked mentions ({len(chunk.entity_links)}): {surface_forms}{" ..." if len(chunk.entity_links) > 5 else ""}')
    print(f'    prefixes seen: {prefixes}')
    text = chunk.text.replace("\n", " ")
    print(f'    text: {text[:260]}{"..." if len(text) > 260 else ""}')
    print()

## 6. Attach chunks to shelves

`fs.attach()` is the bridge phase between Layer A projection and everything downstream. It walks every chunk, resolves each FOODON id to a surviving shelf via the same direct → collapsed → lifted cascade the §5d audit previews, then writes:

- **Neo4j**: `(:Chunk)-[:ATTACHED_TO {lifted_from: [foodon_id,...]}]->(:Shelf)` edges. `lifted_from` is empty for direct attachments (chunk linked the shelf's own term); non-empty when the chunk reached the shelf via collapse or by lifting through ancestors.
- **Elastic**: each chunk's `shelf_ids` keyword-array gets the union of its resolved shelves, so retrieval can filter `terms { shelf_ids: [...] }` without crossing the graph.

Attachment is **nearest surviving ancestor only** (one shelf per FOODON id, the deepest match), facet-aware (`route_link_to_facet` keeps a `medical condition` mention out of the foods walk), and idempotent (re-running with the same projection produces the same edges and the same `lifted_from` payloads). Orphan FOODON ids whose ancestry can't reach any real shelf fall through to the synthetic facet root (`facet:foods`, etc.).

In [ ]:
meta = fs.attach()
print(meta)

# Per-shelf attachment counts straight from Neo4j. Order matches the §5d
# audit's "top 10 foods shelves by chunk_count" so any discrepancy between
# projection-time `chunk_count` and post-attach edge count flags drift —
# typically a chunk whose deepest surviving ancestor differs from the term
# that drove support during projection (e.g. when collapse rewrites the
# parent chain after support was already counted).
from collections import Counter

attach_counts: Counter = Counter()
lifted_edges = 0
direct_edges = 0
with fs.graph_store._driver.session() as session:  # type: ignore[attr-defined]
    result = session.run(
        """
        MATCH (c:Chunk)-[r:ATTACHED_TO]->(s:Shelf)
        WHERE s.facet = 'foods'
        RETURN s.shelf_id AS sid, s.label AS label,
               count(*) AS n,
               sum(CASE WHEN size(r.lifted_from) = 0 THEN 1 ELSE 0 END) AS direct,
               sum(CASE WHEN size(r.lifted_from) > 0 THEN 1 ELSE 0 END) AS lifted
        ORDER BY n DESC
        LIMIT 12
        """
    )
    print(f'\ntop foods shelves by attached chunks (graph edges):')
    for row in result:
        attach_counts[row['sid']] = row['n']
        direct_edges += row['direct']
        lifted_edges += row['lifted']
        print(f'  {row["label"][:40]:40s}  {row["n"]:>5d}  '
              f'(direct={row["direct"]}, via lifted_from={row["lifted"]})')

print(f'\nedge breakdown in top 12: direct={direct_edges}, lifted/collapsed={lifted_edges}')

# Pick one chunk and show what it landed on — concrete confirmation that
# the in-Elastic shelf_ids denorm matches the graph edges, and that
# lifted_from payloads survive the round-trip.
sample_chunk = next(
    (c for c in fs.chunk_store.scan() if c.shelf_ids), None
)
if sample_chunk is not None:
    print(f'\nsample chunk {sample_chunk.chunk_id!r}:')
    print(f'  foodon_ids on chunk:    {sample_chunk.foodon_ids[:6]}')
    print(f'  shelf_ids (Elastic):    {sample_chunk.shelf_ids}')
    with fs.graph_store._driver.session() as session:  # type: ignore[attr-defined]
        result = session.run(
            """
            MATCH (c:Chunk {chunk_id: $cid})-[r:ATTACHED_TO]->(s:Shelf)
            RETURN s.label AS label, s.shelf_id AS sid, r.lifted_from AS lifted_from
            """,
            cid=sample_chunk.chunk_id,
        )
        print(f'  Neo4j edges:')
        for row in result:
            mode = 'direct' if not row['lifted_from'] else f'lifted_from={row["lifted_from"]}'
            print(f'    -> {row["label"]!r:40s}  ({mode})')

## 6b. Audit + quality report

Two checks, two different questions:

- **`fs.audit()`** — *is the graph correctly built?* Five invariants across the chunk store, entity store, and graph store. Critical failures mean Layer B will build on a broken Layer A. Use this to gate downstream work.
- **`fs.quality_report(facet='foods')`** — *is the graph good?* A domain-expert-facing report with five sections (top shelves, hierarchy walkthrough, suspicious shelves, canonical vocabulary check, random chunk sample) formatted as Markdown for in-notebook review.

Audit answers yes/no with hard thresholds; quality is a conversation a nutritionist reads top-to-bottom in ~30 minutes. The quality report is intentionally NOT a single score — quality is multidimensional, and collapsing it would hide the failures that matter.

In [ ]:
# Audit first — invariants. If this fails, the rest is on shaky ground.
audit = fs.audit()
print(audit)
print()
if audit.critical_failures:
    print(f'⚠ {len(audit.critical_failures)} critical invariant(s) failing. '
          'Inspect audit.critical_failures[i].sample for examples.')

# Then quality. Long Markdown — render via IPython for in-notebook readability.
from IPython.display import Markdown

quality = fs.quality_report(facet='foods', top_n=15, sample_size=20, seed=0)
Markdown(str(quality))

## 7. Inspect


In [ ]:
# A handful of chunks with their attached mentions / foodon ids.
for c in fs.chunk_store.scan()[:3]:
    print(f"\n[{c.chunk_id}] {c.source_type} — {c.text[:80]}...")
    print(f"  mentions:   {[m.text for m in c.mentions[:6]]}")
    print(f"  foodon_ids: {c.foodon_ids[:6]}")
    other = sorted({ln.ontology_id for ln in c.entity_links if not ln.ontology_id.startswith('FOODON:')})
    print(f"  other_links: {other[:6]}")

# BM25 round-trip against Elastic.
hits = fs.chunk_store.search("olive oil", k=3, use_vector=False)
print("\n'olive oil' top 3:")
for h in hits:
    print(f"  {h.chunk_id}  {h.text[:80]}")

## 8. Visualize

`fs.viz` builds typed graphs (`VizGraph`) at five abstraction levels and renders
them via pluggable backends. Requires the `[viz]` extra (`pip install -e '.[viz]'`).

| Level | Method                                  | What it shows |
| ----- | --------------------------------------- | ------------- |
| L0    | `fs.viz.entity_histogram(prefix=, k=)`  | Top-k entities by chunk_count (stats) |
| L1    | `fs.viz.entity_neighborhood(ontology_id)` | One entity + mentioning chunks + co-entities |
| L2    | `fs.viz.shelf(shelf_id)`                | One Layer A shelf + themes + chunks |
| L3    | `fs.viz.backbone(facet=)`               | Whole Layer A/B/C backbone |
| L4    | `fs.viz.ontology_subtree(ontology_id)`  | FoodOn ancestors + descendants |

Renderer choice via `.render(backend, output=...)`:
- `"pyvis"`      → interactive HTML (best in-notebook).
- `"cytoscape"`  → self-contained Cytoscape.js HTML, no Python deps beyond stdlib.
- `"graphviz"`   → static SVG / PNG (needs `dot` binary).
- `"matplotlib"` → bars / heatmaps over node weights (ignores edges; perfect for L0).

In [ ]:
# L0 — top entities, rendered with matplotlib (inline figure).
fig = fs.viz.entity_histogram(k=20).render("matplotlib")

In [ ]:
# L1 — entity neighborhood. Render to an HTML string and inline it via
# IPython.display.HTML(). Earlier we wrote the HTML to a file and embedded
# via IFrame(src=...) — VS Code's notebook webview can't fetch local files
# from a relative path, so the iframe stayed empty. Inline HTML works in
# every notebook environment.
from IPython.display import HTML

top = fs.entities.list(prefix='FOODON', k=1)
anchor_id = top[0].ontology_id if top else 'FOODON:03309927'

html = fs.viz.entity_neighborhood(anchor_id, max_chunks=8).render('cytoscape')
HTML(html)

In [ ]:
# L4 — ontology subtree around a single FoodOn id. Inline HTML, same
# recipe as L1.
from IPython.display import HTML

# Top entities by chunk_count are often leaves in the FoodOn hierarchy (e.g.
# `food calorie datum`) — picking one of those produces a 1-edge subtree.
# Walk the top-50 and prefer anchors with real downward structure: ≥ 3
# descendants ideally, then ≥ 1, then whatever's left.
_onto = fs.ontology
_top = [e.ontology_id for e in fs.entities.list(prefix='FOODON', k=50)]
_resolved = [(oid, len(_onto.id_to_descendants(oid)))
             for oid in _top if _onto.get(oid) is not None]
subtree_id = next((oid for oid, n in _resolved if n >= 3),
                  next((oid for oid, n in _resolved if n >= 1),
                       _resolved[0][0] if _resolved else anchor_id))
if subtree_id != anchor_id:
    n_desc = dict(_resolved).get(subtree_id, 0)
    print(f'L4 anchor: {subtree_id} ({n_desc} descendants) — L1 anchor {anchor_id} was a leaf or unresolved')

rg_subtree = fs.viz.ontology_subtree(subtree_id, max_descendants=20)
print(rg_subtree)

html = rg_subtree.render('cytoscape')
HTML(html)

In [ ]:
# L2 (Layer A backbone, foods facet). Inline HTML, same recipe as L1.
from IPython.display import HTML

rg_backbone = fs.viz.backbone(facet='foods')
print(rg_backbone)

html = rg_backbone.render('cytoscape')
HTML(html)

---

## Under the hood (appendix)

The three cells above are everything an end-user needs. The appendix below is
for contributors and shows the same pipeline **without** pre-computed NER/NEL —
i.e. running GLiNER + HNSW from scratch. Skip it unless you're debugging the
NER side.

### A1. Run GLiNER + HNSW directly

Requires `pip install -e '.[annotate]'`. First call downloads the GLiNER bio
model (~1.5 GB) and the BioLORD encoder (~440 MB), then builds and caches the
FoodOn HNSW index under `data/`. Subsequent runs are fast.

In [ ]:
import importlib.util as _u

needs = [pkg for pkg in ("gliner", "hnswlib", "sentence_transformers") if not _u.find_spec(pkg)]
if needs:
    print("Skipping — missing:", needs, "\nInstall with:  pip install -e '.[annotate]'")
else:
    fs_g = FoodScholar.from_config({
        "corpus": {"chunks_path": str(CORPUS_DIR)},
        "ontology": {
            "foodon_path": str(FOODON_OWL),
            "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
            "prefix_filter": ["FOODON:"],
        },
        "storage": {
            "chunk_store": {"backend": "memory"},
            "graph_store": {"backend": "memory"},
        },
    })
    one_file = next(iter(sorted(CORPUS_DIR.glob("*.csv"))))
    fs_g.ingest(one_file)  # no nel_dir → GLiNER + HNSW path
    c = fs_g.chunk_store.scan()[0]
    print(c.chunk_id, "mentions:", [m.text for m in c.mentions[:6]])

### A2. Inspect the graph

Layer B/C builders are still stubs — this appendix shows the `fs.graph` API
the builders will write through. Run it against the in-memory facade so it
doesn't depend on the live services.

In [ ]:
fs_local = FoodScholar.in_memory()
fs_local.graph.add_shelf(shelf_id="s-med", label="Mediterranean diet",
                         facet="dietary_patterns", depth=0)
fs_local.graph.add_theme(theme_id="t-olive", label="Olive oil benefits",
                         shelf_ids=["s-med"], discovered_by="leiden",
                         discovery_version="nb")
shelf = fs_local.graph.shelf("s-med")
print("shelf :", shelf.label, "| facet:", shelf.facet)
print("themes:", [t.label for t in shelf.themes()])